# 👥 Notebook 4: Patient Risk Stratification via Clustering
**Direct Mapping to Experiment 10:** Benchmarking K-Means, Agglomerative (Hierarchical), and Gaussian Mixture Models (GMM) using Silhouette & Davies-Bouldin metrics.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score
import sys
sys.path.append('..')
from src.preprocessing import prepare_data


### 1. Optimal Cluster Search via Silhouette Analysis


In [ ]:
data = prepare_data('../data/heart_failure_clinical_records.csv')
X = data['X_train_scaled']

silhouettes = []
k_range = range(2, 7)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    silhouettes.append(silhouette_score(X, labels))

plt.figure(figsize=(7, 4))
plt.plot(k_range, silhouettes, 'go-')
plt.title('Silhouette Score vs k (Exp 10)')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.show()


### 2. Multi-Algorithm Clustering Benchmark


In [ ]:
k = 3
km = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X)
agg = AgglomerativeClustering(n_clusters=k).fit_predict(X)
gmm = GaussianMixture(n_components=k, random_state=42).fit_predict(X)

comparison = [
    {'Algorithm': 'K-Means', 'Silhouette': silhouette_score(X, km), 'Davies-Bouldin': davies_bouldin_score(X, km)},
    {'Algorithm': 'Agglomerative Clustering', 'Silhouette': silhouette_score(X, agg), 'Davies-Bouldin': davies_bouldin_score(X, agg)},
    {'Algorithm': 'Gaussian Mixture Model', 'Silhouette': silhouette_score(X, gmm), 'Davies-Bouldin': davies_bouldin_score(X, gmm)}
]
pd.DataFrame(comparison)


### 3. Clinical Risk Profiling of Discovered Cohorts


In [ ]:
df_profile = data['X_train'].copy()
df_profile['Cohort'] = agg
df_profile['Mortality_Rate'] = data['y_train']

summary = df_profile.groupby('Cohort').agg({
    'age': 'mean',
    'ejection_fraction': 'mean',
    'serum_creatinine': 'mean',
    'Mortality_Rate': 'mean'
})
print("Clinical Risk Phenotypes:")
summary
